## 4.3 自动微分 - autograd的小问题&detach()

#### 1. 常见问题

##### 1.1 梯度累加问题：

现象：
* .grad 越来越大，或者训练越来越不稳定。

原因：
* PyTorch 的设计是：每次 backward 得到的梯度会加到 .grad 上，而不是覆盖。

正确做法：
* 每一轮训练开始前清零：`w.grad.zero_()` 或者 `optimizer.zero_grad()`

##### 1.2 参数更新的计算过程不应该写入到计算图中（内存爆炸）

错误写法：
* `w -= lr * w.grad`
* 这会让更新也进入计算图，导致：
    * 图越来越复杂
    * 下一次 backward 混乱
    * 内存爆炸

正确写法:
* 用 torch.no_grad() 包裹更新：
* ` with torch.no_grad():
    w -= lr * w.grad `

⚠️注意：
* 虽然参数更新的计算过程不应该在计算图中
* 但是参数 w 不能被重新赋值为新的变量，而是沿用w变量，做原地修改

##### 1.3 为什么有时会报错：Trying to backward through the graph a second time?

现象:
* 对同一个 loss 调了两次 backward()，或保存了某个中间结果后又 backward，出现类似错误：
* Trying to backward through the graph a second time…

原因：
* 默认情况下：
    * backward 一次后，计算图就被释放（为了省内存）
    * 你再 backward，就找不到图了

解决方法：
1. 方法A：你确实需要二次 backward（高级用法）
    * `loss.backward(retain_graph=True)`
2. 方法B（更常见）：你只是写法不对
    * 一般训练中 每轮只 backward 一次

##### 1.4  为什么显存/内存越用越多？（计算图被你“存起来了”)

典型错误：
* 你在循环里把带计算图的 tensor 存进 list
* 这样会导致：
    * 每一轮的计算图都被 losses 引用着，无法释放 → 内存越来越大

正确做法：
* 你只存数值：`losses.append(loss.item())  # ✅ 只存 Python 数值`
* 或者存“断开图”的 tensor（后面讲 detach）：`losses.append(loss.detach())`

#### 2. detach()：停止梯度传播的关键工具 

##### 2.1 detach 到底做了什么？

`y_detached = y.detach()`

含义是：
* 返回一个新的 Tensor，它和 y 共享数据，但 不再被 Autograd 追踪
* 从这个点开始，梯度不会再往前传


##### 2.2 detach 示例：证明它会“断开梯度链路”

In [1]:
import torch
a = torch.tensor(2.0, requires_grad=True)
b = a * a
c = b.detach() + 3 # detach()方法返回一个新的tensor，和原来的tensor共享数据，但不记录梯度信息

c.backward() # ❗️此时c不再是计算图的一部分，所以调用backward()方法时会报错

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

##### 2.3 detach 最常见使用场景
1. 记录日志/画图，但不想占用计算图
    * `loss_history.append(loss.detach())` 或者：
    * `loss_history.append(loss.item())`
2. 生成“伪标签 / target”，不让梯度回传
    * 例如在一些训练策略里：
    * 用模型输出当成 target
    * 但 target 不应该影响模型参数